In [ ]:
import re
import pandas as pd


In [1]:
import re

In [2]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', "", text)
    return text

In [3]:
processed = ""
with open("Harry_Potter_all_books_preprocessed.txt", 'r', encoding='utf-8') as f:
    for lines in f:
        processed += preprocess(lines)
    f.close()


In [4]:
with open("cleaned.txt", 'w', encoding='utf-8') as f:
    f.write(processed)

In [5]:
def _words_counts(text) -> dict:
    text = preprocess(text)
    word_counts = {}
    for word in text.split():
        word_counts[str(word)] = word_counts[str(word)]+ 1 if str(word) in word_counts else 1
    return word_counts

def individual_words_counts(file_path):
    text = ""
    with open(file_path,'r',encoding='utf-8') as f:
        for line in f:
            text += line
    return _words_counts(text)

def build_ngram(text ,n):
    ng = []
    ngrams = []
    for words in text.split():
        ng.append((words))
    for i in range(len(ng) -n + 1):
        ngrams.append(tuple((ng[i:i+n])))

    return ngrams

def get_count(word, ngram):
    '''Word is a tuple   of the words in a sentence'''
    count = 0
    for w in ngram:
        if w == word:
            count+= 1

    return count

def get_prob(word1, word2):
    '''Word1 and word2 are supposed to be  dictipnaries'''
    print(len(word1), len(word2))
    return get_count(word1, build_ngram(test, len(word1))) / get_count(word2, build_ngram(test, len(word2)))

In [6]:
from collections import defaultdict

def build_ngram_counts_efficient(text, max_n):
    
    tokens = text.split()
    counts = {n: defaultdict(int) for n in range(1, max_n + 1)}

    for i in range(len(tokens)):
        for n in range(1, max_n + 1):
            if i + n <= len(tokens):
                gram = tuple(tokens[i: i + n])
                counts[n][gram] += 1 

    return counts

In [7]:
def build_ngram_probs(text, k):

    counts = build_ngram_counts_efficient(text, k+1)
    probs = {}
    for n in range(1, k+1):
        context_next_counts = defaultdict(lambda : defaultdict(int))

        for gram , cnt in counts[n+1].items():
            context = gram[:n]
            next_word = gram[n]
            context_next_counts[context][next_word] += cnt

            # print(context , next_word , cnt, context_next_counts[context][next_word])
        probs[n]= {}

        for context , next_word_counts in context_next_counts.items():
            # print(next_word_counts.values())
            total = sum(next_word_counts.values())
            probs[n][context] = {
                word: cnt / total
                for word, cnt in sorted(
                    next_word_counts.items(), key=lambda x: -x[1]
                )
            }

    return probs

    # print(context_next_counts.items())


In [8]:
probs_5 = build_ngram_probs(processed, 5)

In [ ]:
# probs_5

In [9]:
#probability of next word =  probs_5[depth ][(dictionary of words)] [next word ]

In [21]:
def predict_next_words(initial_word_dict):
    length = len(initial_word_dict)
    # print(length)
    number_of_words = 6 - length
    next_words = ""
    new_word_dict = ('',)
    for i in range(number_of_words):
        next = list((probs_5[i+length])[initial_word_dict])[0]

        # print(next)
        new_word_dict += (next,)
        # next_words = next_words + " " + next


    for words in new_word_dict:
        next_words = next_words + " " + words

    return next_words

def predict_more(initial_word_dict):
    next_words = ""
    # t = 50
    # while (t > 0):
        # print(len(initial_word_dict))
        # if (len(initial_word_dict) >= 5):
        #     initial_word_dict = initial_word_dict[1:]

    # words = predict_next_words(initial_word_dict)
    return predict_next_words(initial_word_dict)
        # print((next_words))
    paragraph =  paragraph + " "  + words
        # initial_word_dict = words_dict
        # next =(list(initial_word_dict))[-1] 
        # next_words = next_words + " " + next
        # print((list(initial_word_dict))[-1])

        # t = t-1
    # print(next_words, end=" ")


def predict_from_paragraph(paragraph):
    last_5 = paragraph.split(" ")[-5:]

    last_5_tup = tuple(last_5)

    return predict_more(last_5_tup)

# def predict_next_word()




# predict_next_words(initial_word)
# t= 50
# while (t>0):
#     paragraph = paragraph + " " + predict_from_paragraph(paragraph)
#     print(paragraph)
#     t -= 1
# print(paragraph)
# predict_from_paragraph(initial_word)
# print(predict_next_words(initial_word))


In [11]:
def get_next(paragraph):
    last_ = paragraph.split(" ")[-5:]

    last_tup = tuple(last_)
    # print(last_tup)
    length_of_context = len(last_tup)
    # print(length_of_context)
    next = list((probs_5[length_of_context])[last_tup])[0]

    return next

In [12]:
texts = "harry potter and i shall"

In [13]:
import time
t = 50
while (t>0):
    # print(texts)
    next = get_next(texts)
    texts = texts + " " + next
    # print(texts)

    print(next, end = " ")
    t-=1
    time.sleep(1)

be at these words seemingly in response to them a sudden wail sounded a terrible drawnout cry of misery and pain many of those at the table looked downward startled for the sound had seemed to issue from below their feet 

KeyboardInterrupt: 

In [14]:
maax = -1
word = ""
for key, value in probs_5[1].items():
    if len(value) > maax:
        maax = len(value)
        word = key

print(f" {word}, : {maax}")

 ('the',), : 5623
